# News Universe: analisis offline 21–27 September 2026

**Pertanyaan:** berita lokal Sectors.app apa yang menautkan emiten/sektor, tema apa yang benar-benar tertulis, dan jalur hipotesis mana yang layak diperiksa berikutnya? Ini analisis deskriptif berbasis aturan, bukan model kausal, rekomendasi investasi, atau prediksi return.

Roadmap: provenance → cakupan/kualitas → analisis setiap berita → prioritas Top 5 → batas bukti dan langkah berikutnya. Tidak ada panggilan jaringan.

In [1]:
from pathlib import Path
import sys
import pandas as pd
pd.set_option('display.max_rows', None)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/raw/sectors/news').exists():
    raise FileNotFoundError('Jalankan dari root Hackathon Sectors atau lampirkan folder data/raw/sectors/news beserta metadata snapshot dan modul src/idx_evidence_lab.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from idx_evidence_lab.market_data import load_news_universe
from idx_evidence_lab.news_analysis import analyze_news_article
print('Root:', PROJECT_ROOT.resolve())
print('Analisis offline; tidak ada API/model yang dipanggil.')

Root: /Users/rasyaaudreabramantyawijaya/Downloads/Hackathon Sectors
Analisis offline; tidak ada API/model yang dipanggil.


## 1. Cakupan dan validasi sumber
Loader hanya mempertahankan snapshot Sectors.app yang metadata, HTTP status, endpoint dan SHA-256-nya lolos pemeriksaan. Duplikasi snapshot dihapus dengan kunci waktu–judul–sumber. Baris tanpa body tetap terlihat sebagai partial bila lolos validasi dasar.

In [2]:
dataset = load_news_universe(PROJECT_ROOT)
articles = dataset['articles']
assert articles, 'Tidak ada artikel valid pada snapshot lokal.'
start = min(a['timestamp'] for a in articles)
end = max(a['timestamp'] for a in articles)
audit = pd.DataFrame([
    {'item':'Provider','value':dataset['provider']},
    {'item':'Verified snapshots','value':dataset['snapshot_count']},
    {'item':'Deduplicated articles','value':len(articles)},
    {'item':'Article timestamp range','value':f'{start} → {end}'},
    {'item':'Snapshot coverage end','value':dataset['coverage_end']},
    {'item':'Data quality','value':dataset['data_quality']},
    {'item':'Issues','value':len(dataset['issues'])},
] )
display(audit)
display(pd.Series([a['timestamp'][:10] for a in articles]).value_counts().sort_index().rename_axis('date').to_frame('articles'))

,item,value
0,Provider,Sectors.app
1,Verified snapshots,7
2,Deduplicated articles,145
3,Article timestamp range,2026-09-21T05:10:00 → 2026-09-27T15:01:00
4,Snapshot coverage end,2026-09-27
5,Data quality,VERIFIED
6,Issues,0


,articles
date,
2026-09-21,27
2026-09-22,22
2026-09-23,42
2026-09-24,33
2026-09-25,3
2026-09-26,12
2026-09-27,6


## 2. Metode offline yang sama dengan prototype
Setiap item dicocokkan dengan kamus frasa yang terlihat di `news_analysis.py`. Ticker/sektor/tema yang muncul dicatat sebagai **observasi/mention**, bukan penyebab. Hubungan tema→sektor/emiten ditampilkan hanya sebagai **hipotesis untuk diuji**. Top 5 adalah prioritas baca deterministik berdasarkan ticker tertaut, istilah material, banyaknya emiten, dan tema; bukan ranking dampak pasar. Tidak ada probabilitas/confidence rekaan.

In [3]:
for article in articles:
    article['offline_analysis'] = analyze_news_article(article)
df = pd.DataFrame([{
    'timestamp': a['timestamp'], 'title': a['title'],
    'symbols': ', '.join(a['symbols']), 'sector': a['sector'],
    'priority_score_rule_only': a['offline_analysis']['priority_score'],
    'observations': ' | '.join(a['offline_analysis']['observations']),
    'drivers_found_in_text': ' | '.join(d['label'] for d in a['offline_analysis']['drivers']) or 'Tidak terdeteksi',
    'hypothesis_paths_not_verified': ' | '.join(a['offline_analysis']['hypotheses']) or 'Tidak cukup dasar teks',
    'priority_reason': ' | '.join(a['offline_analysis']['priority_reasons']),
    'source_domain': a['offline_analysis']['source_domain'],
    'source_file': a['source_file'], 'news_id': a['news_id'],
} for a in articles]).sort_values(['timestamp','news_id'])
print('Seluruh berita, tertua → terbaru:', len(df))
display(df)

Seluruh berita, tertua → terbaru: 145


,timestamp,title,symbols,sector,priority_score_rule_only,observations,drivers_found_in_text,hypothesis_paths_not_verified,priority_reason,source_domain,source_file,news_id
144,2026-09-21T05:10:00,Analysts forecast IHSG weakness on Monday 21 S...,"BULL, KLBF, TLKM, BBCA",energy,7,"Emiten tertaut pada record berita: BULL, KLBF,...",Regulasi / otoritas | Kinerja / prospek usaha,Hipotesis: Regulasi / otoritas → energy; belum...,ticker tertaut langsung | istilah material ter...,market.bisnis.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,4358cea29645cb17db2ef53f
143,2026-09-21T07:40:00,PT Adhi Karya (Persero) Tbk (ADHI) signs Maste...,"ADHI, BMRI, BBRI, BBNI",infrastructures,8,"Emiten tertaut pada record berita: ADHI, BMRI,...",Aksi korporasi / transaksi | Kinerja / prospek...,Hipotesis: Aksi korporasi / transaksi → infras...,ticker tertaut langsung | istilah material ter...,idnfinancials.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,a722248238205c9e5b7049dd
142,2026-09-21T08:38:00,"Today's Stock Recommendations: ANTM, VKTR, and...","ANTM, VKTR, MAPI, BSDE, SCMA, BBYB, ERAA, BIKE...",basic-materials,7,"Emiten tertaut pada record berita: ANTM, VKTR,...",Aksi korporasi / transaksi,Hipotesis: Aksi korporasi / transaksi → basic-...,ticker tertaut langsung | istilah material ter...,finance.detik.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,1ce8134d3310fcede8572d81
141,2026-09-21T08:50:00,"IHSG Forecast to Test 6.633 This Week, with An...","ANTM, MTEL, IMPC",basic-materials,7,"Emiten tertaut pada record berita: ANTM, MTEL,...",Bank Indonesia / suku bunga | Kinerja / prospe...,Hipotesis: Bank Indonesia / suku bunga → basic...,ticker tertaut langsung | istilah material ter...,market.bisnis.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,a62034fd0aea39048f8db9ab
140,2026-09-21T09:10:00,Foreign Investors Net Sold Rp2.9 trillion in t...,"BMRI, BRMS",financials,5,"Emiten tertaut pada record berita: BMRI, BRMS ...",Tidak terdeteksi,Tidak cukup dasar teks,ticker tertaut langsung | menyebut lebih dari ...,bloombergtechnoz.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,8604c0e9a59633188a82a737
139,2026-09-21T11:20:00,PT Adhi Karya Tbk (ADHI) signs Master Restruct...,"ADHI, BMRI, BBRI, BBNI",infrastructures,8,"Emiten tertaut pada record berita: ADHI, BMRI,...",Aksi korporasi / transaksi | Kinerja / prospek...,Hipotesis: Aksi korporasi / transaksi → infras...,ticker tertaut langsung | istilah material ter...,investasi.kontan.co.id,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,d3bc393f4211d04d4d2b89e7
138,2026-09-21T11:43:46,Hery Buys Shares of Charoen Pokphand Indonesia,CPIN,consumer-non-cyclicals,4,Emiten tertaut pada record berita: CPIN | Sekt...,Aksi korporasi / transaksi,Hipotesis: Aksi korporasi / transaksi → consum...,ticker tertaut langsung | istilah material ter...,idx.co.id,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,12f4edb2133abc8b57a45246
137,2026-09-21T12:45:00,IHSG down 0.27% as foreign investors sell shar...,"ANTM, TPIA, BBRI, BBCA",basic-materials,5,"Emiten tertaut pada record berita: ANTM, TPIA,...",Tidak terdeteksi,Tidak cukup dasar teks,ticker tertaut langsung | menyebut lebih dari ...,idnfinancials.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,0d6e9f27d40277983ed80282
136,2026-09-21T13:17:00,PT Amman Mineral Internasional Tbk (AMMN) repo...,AMMN,basic-materials,6,Emiten tertaut pada record berita: AMMN | Sekt...,Komoditas | Kinerja / prospek usaha,Hipotesis: Komoditas → basic-materials; belum ...,ticker tertaut langsung | istilah material ter...,bloombergtechnoz.com,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,ce80a30d0a44004f2386853b
135,2026-09-21T13:43:49,Ubs Ag Ldn Branch A/C Client Buy Shares of Cha...,CPIN,consumer-non-cyclicals,3,Emiten tertaut pada record berita: CPIN | Sekt...,Tidak terdeteksi,Tidak cukup dasar teks,ticker tertaut langsung,idx.co.id,data/raw/sectors/news/idx_lq45_2026-09-21_2026...,0543b52f45cf89fd2174de48


## 3. Top 5 untuk dipindai dalam 10 menit
Skor hanya urutan heuristik lokal. Periksa alasan per berita; urutan tidak membuktikan materialitas harga atau sebab-akibat.

In [4]:
top5 = sorted(articles, key=lambda a: (-a['offline_analysis']['priority_score'], -pd.Timestamp(a['timestamp']).timestamp(), a['news_id']))[:5]
display(pd.DataFrame([{'rank':i+1,'date':a['timestamp'],'title':a['title'],'symbols':', '.join(a['symbols']),'rule_score':a['offline_analysis']['priority_score'],'why':'; '.join(a['offline_analysis']['priority_reasons'])} for i,a in enumerate(top5)]))

,rank,date,title,symbols,rule_score,why
0,1,2026-09-26T20:24:00,PT Merdeka Copper Gold Tbk (MDKA) reports US$1...,"MDKA, MBMA, EMAS",10,ticker tertaut langsung; istilah material tert...
1,2,2026-09-23T17:00:00,PT Astra International Tbk (ASII) realizes Rp1...,"ASII, UNTR",10,ticker tertaut langsung; istilah material tert...
2,3,2026-09-26T06:00:00,Popular news roundup highlights PT Pioneerindo...,"PTSP, TRUK, ENRG, BBCA",9,ticker tertaut langsung; istilah material tert...
3,4,2026-09-23T14:28:00,PT Astra International Tbk (ASII) reports firs...,"ASII, UNTR",9,ticker tertaut langsung; istilah material tert...
4,5,2026-09-23T07:57:00,"BRI Danareksa Sekuritas recommends BIPI, RMKE ...","BIPI, RMKE, CUAN",9,ticker tertaut langsung; istilah material tert...


## 4. Kesimpulan dan batas
Analisis ini hanya memetakan teks dan metadata dari snapshot yang lolos provenance. Kata BI/menteri keuangan/komoditas hanya muncul sebagai driver bila literal terdeteksi dalam artikel; ketiadaan deteksi bukan bukti tidak ada pengaruh. Untuk menguji hipotesis perlu data harga/event point-in-time, definisi jendela dampak, baseline, dan validasi—belum dihitung di sini. Relationship map UI memakai mention solid dan hypothesis dashed.

**Backward trace:** ringkasan/Top 5 → aturan heuristik v1 → artikel/timestamp/ticker/tema → snapshot Sectors.app terverifikasi. Ini tidak membuktikan hubungan sebab-akibat, dampak return, atau kelayakan investasi. Reproducibility: loader dan kamus lokal, tanpa seed/model/API; cakupan dan SHA-256 snapshot ada di metadata sumber.